# Training a CNN on MNIST with PyTorch Lightning

This notebook demonstrates how to train a simple, efficient Convolutional Neural Network (CNN) with residual connections on the MNIST dataset using PyTorch Lightning. The configuration is tuned for ≥99.7% accuracy.

In [12]:
!pip install pytorch-lightning torch torchvision datasets wandb pillow scikit-learn


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [13]:
from PIL import ExifTags, Image
Image.ExifTags = ExifTags  # Hack to bypass broken import

In [ ]:
# --- Config Setup ---
import os

def notebook_id_from_title():
    # Dummy implementation for notebook ID
    return "mnist-cnn-pl"

def setup_config():
    # General Settings
    seed = 42
    n_epochs = 100
    batch_size = 2056
    # Optimizer Settings
    learning_rate = 1e-3
    weight_decay = 0
    warmup_ratio = 0
    max_grad_norm = 0
    # Model Architecture
    hidden_size = 128
    nonlinearity = 'relu'
    weight_init = 'kaiming'
    # Data Settings
    sequence_length = 1
    train_size = 0.9167  # 55000/60000
    val_size = 0.0833    # 5000/60000
    input_size = 1
    output_size = 10
    # Softcoded new params
    label_smoothing = 0.1
    dropout = 0.1
    early_stopping_patience = 7
    early_stopping_min_delta = 1e-4
    use_mixed_precision = True
    use_swa = True
    swa_lrs = 1e-2
    activation = "silu"  # or "ReLU", "LeakyReLU", etc.

    #os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'sequence_length': sequence_length,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size,
        'nonlinearity': nonlinearity,
        'weight_init': weight_init,
        'weight_decay': weight_decay,
        'warmup_ratio': warmup_ratio,
        'max_grad_norm': max_grad_norm,
        'train_size': train_size,
        'val_size': val_size,
        'label_smoothing': label_smoothing,
        'dropout': dropout,
        'early_stopping_patience': early_stopping_patience,
        'early_stopping_min_delta': early_stopping_min_delta,
        'use_mixed_precision': use_mixed_precision,
        'use_swa': use_swa,
        'swa_lrs': swa_lrs,
        'activation': activation
    }

CONFIG = setup_config()

In [15]:
import torch
torch.set_float32_matmul_precision('high')  # or 'medium'

In [16]:
import wandb
wandb.login()

True

In [ ]:
from torch import nn
from torchvision.datasets import MNIST

def get_activation():
    from torch import nn
    act = CONFIG.get('activation', 'silu')
    if act.lower() == "silu": return nn.SiLU()
    elif act.lower() == "relu": return nn.ReLU()
    elif act.lower() == "leaky_relu": return nn.LeakyReLU()
    elif act.lower() == "tanh": return nn.Tanh()
    elif act.lower() == "sigmoid": return nn.Sigmoid()
    else: raise ValueError(f"Unsupported activation: {act}")
    
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=None):
        super().__init__()

        dropout = CONFIG['dropout']

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.act1 = get_activation()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if dropout: self.dropout = nn.Dropout2d(dropout)
        self.downsample = None

        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, 0, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        dropout = CONFIG['dropout']

        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if dropout: out = self.dropout(out)
        if self.downsample is not None: identity = self.downsample(x)
        out += identity
        out = get_activation()(out)
        return out


In [18]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
import pytorch_lightning as pl

class MNISTDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=None, num_workers=2):
        super().__init__()
        self.batch_size = batch_size if batch_size is not None else CONFIG['batch_size']
        self.num_workers = num_workers
        self.train_transform = transforms.Compose([
            transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.RandomRotation(degrees=15),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
            transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random')
        ])
        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    def prepare_data(self):
        MNIST(root="./data", train=True, download=True)
        MNIST(root="./data", train=False, download=True)

    def setup(self, stage=None):
        mnist_full = MNIST(root="./data", train=True, transform=self.train_transform)
        total = len(mnist_full)
        train_size = int(total * CONFIG['train_size'])
        val_size = total - train_size
        self.mnist_train, self.mnist_val = torch.utils.data.random_split(
            mnist_full, [train_size, val_size], generator=torch.Generator().manual_seed(CONFIG['seed'])
        )
        self.mnist_val.dataset.transform = self.test_transform
        self.mnist_test = MNIST(root="./data", train=False, transform=self.test_transform)

    def train_dataloader(self):
        return DataLoader(self.mnist_train, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.mnist_val, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.mnist_test, batch_size=self.batch_size, num_workers=self.num_workers, pin_memory=True)

dm = MNISTDataModule(batch_size=CONFIG['batch_size'])

In [19]:
import torch.nn.init as init

class LitCNN(pl.LightningModule):
    def __init__(self, lr=None):
        super().__init__()

        self.save_hyperparameters()

        dropout = CONFIG['dropout']
        label_smoothing = CONFIG['label_smoothing']

        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32),
            get_activation()
        )
        self.layer1 = ResidualBlock(32, 64, stride=1, dropout=dropout)
        self.layer2 = ResidualBlock(64, 128, stride=2, dropout=dropout)
        self.layer3 = ResidualBlock(128, 256, stride=2, dropout=dropout)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            get_activation(),
            nn.Dropout(CONFIG['dropout']),
            nn.Linear(128, 10)
        )
        self.criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.validation_step_outputs = []
        self.apply(self._init_weights)

    def _init_weights(self, m):
        # Map config activation to PyTorch nonlinearity string
        nonlinearity = CONFIG['activation'].lower()
        if nonlinearity == "silu": nonlinearity = "relu"

        if isinstance(m, nn.Conv2d):
            init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity=nonlinearity)
            if getattr(m, "bias", None) is not None:
                init.constant_(m.bias, 0)
        elif isinstance(m, nn.Linear):
            init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity=nonlinearity)
            if m.bias is not None:
                init.constant_(m.bias, 0)
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            if hasattr(m, "weight") and m.weight is not None:
                init.constant_(m.weight, 1)
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x)
        x = self.fc(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        self.log("val_acc", acc, prog_bar=True, sync_dist=True)
        self.validation_step_outputs.append({"preds": preds.detach().cpu(), "targets": y.detach().cpu()})
        misclassified = preds != y
        return {
            "misclassified_images": x[misclassified].detach().cpu(),
            "misclassified_preds": preds[misclassified].detach().cpu(),
            "misclassified_labels": y[misclassified].detach().cpu()
        }

    def on_validation_epoch_end(self):
        # --- Confusion matrix logging ---
        import numpy as np
        import wandb
        import torch
        # Gather all preds and targets
        all_preds = []
        all_targets = []
        for out in self.validation_step_outputs:
            all_preds.append(out["preds"])
            all_targets.append(out["targets"])
        if all_preds and all_targets:
            all_preds = torch.cat(all_preds).numpy()
            all_targets = torch.cat(all_targets).numpy()
            # Compute confusion matrix
            from sklearn.metrics import confusion_matrix
            cm = confusion_matrix(all_targets, all_preds, labels=np.arange(10))
            # Log to wandb
            if self.logger is not None and hasattr(self.logger, "experiment"):
                self.logger.experiment.log({
                    "confusion_matrix": wandb.plot.confusion_matrix(
                        probs=None,
                        y_true=all_targets,
                        preds=all_preds,
                        class_names=[str(i) for i in range(10)]
                    ),
                    "epoch": self.current_epoch
                })
        # Clear for next epoch
        self.validation_step_outputs.clear()
        # --- Existing misclassified logging ---
        # Aggregate misclassified samples from all batches
        misclassified_images = []
        misclassified_preds = []
        misclassified_labels = []
        for out in self.trainer.callback_metrics.get("validation_step_outputs", []):
            if out is not None:
                misclassified_images.append(out["misclassified_images"])
                misclassified_preds.append(out["misclassified_preds"])
                misclassified_labels.append(out["misclassified_labels"])
        if misclassified_images:
            import wandb
            import torch
            images = torch.cat(misclassified_images)
            preds = torch.cat(misclassified_preds)
            labels = torch.cat(misclassified_labels)
            # Limit to 16 samples for logging
            n_samples = min(16, images.size(0))
            images = images[:n_samples]
            preds = preds[:n_samples]
            labels = labels[:n_samples]
            # Prepare wandb.Image objects
            img_list = []
            for i in range(n_samples):
                img = images[i].squeeze().numpy()
                caption = f"pred: {preds[i].item()}, label: {labels[i].item()}"
                img_list.append(wandb.Image(img, caption=caption))
            if self.logger and hasattr(self.logger, "experiment"):
                self.logger.experiment.log({"bad_classifications": img_list, "epoch": self.current_epoch})

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log("test_loss", loss)
        self.log("test_acc", acc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr if hasattr(self.hparams, 'lr') and self.hparams.lr is not None else CONFIG['learning_rate'],
            weight_decay=CONFIG['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }
    
model = LitCNN(lr=CONFIG['learning_rate'])

In [20]:
import time
from pytorch_lightning.callbacks import Timer
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import LearningRateMonitor

class EpochTimeLogger(pl.Callback):
    def on_train_epoch_start(self, trainer, pl_module):
        self.epoch_start_time = time.time()

    def on_train_epoch_end(self, trainer, pl_module):
        epoch_time = time.time() - self.epoch_start_time
        # Log to wandb if logger is set
        if trainer.logger is not None and hasattr(trainer.logger, "experiment"):
            trainer.logger.experiment.log({"epoch_time_sec": epoch_time, "epoch": trainer.current_epoch})

trainer_callbacks = [
    Timer(), 
    EpochTimeLogger(), 
    EarlyStopping(
        monitor="val_loss",
        patience=CONFIG['early_stopping_patience'],
        min_delta=CONFIG['early_stopping_min_delta'],
        mode="min",
        verbose=True,
        strict=True
    ), 
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min",
        save_last=True,
        dirpath="checkpoints",
        filename="mnist-cnn-{epoch:02d}-{val_loss:.2f}"
    ), 
    LearningRateMonitor(logging_interval="epoch")
]

if CONFIG['use_swa']:
    from pytorch_lightning.callbacks import StochasticWeightAveraging
    swa_callback = StochasticWeightAveraging(swa_lrs=CONFIG['swa_lrs'])
    trainer_callbacks.append(swa_callback)

In [ ]:
from pytorch_lightning.loggers import WandbLogger

num_gpus = torch.cuda.device_count()

trainer = pl.Trainer(
    devices=num_gpus,
    accelerator="auto",
    strategy="auto",
    benchmark=True,
    max_epochs=CONFIG['n_epochs'],
    logger=WandbLogger(project="mnist-cnn-pl"),
    callbacks=trainer_callbacks,
    precision="16-mixed" if CONFIG['use_mixed_precision'] else 32
)
trainer.fit(model, datamodule=dm)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/3
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/3
Initializing distributed: GLOBAL_RANK: 2, MEMBER: 3/3
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 3 processes
----------------------------------------------------------------------------------------------------



/usr/local/lib/python3.10/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 2 - CUDA_VISIBLE_DEVICES: [0,1,2]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1,2]

  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | stem        | Sequential        | 352    | train
1 | layer1      | ResidualBlock     | 57.7 K | train
2 | layer2      | ResidualBlock     | 230 K  | train
3 | layer3      | ResidualBlock     | 919 K  | train
4 | global_pool | AdaptiveAvgPool2d | 0      | train
5 | fc          | Sequential        | 34.4 K | train
6 | criterion   | CrossEntropyLoss  | 0      | train
----------------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.967     Total estimated model params size (MB)
43        Modules in train mode
0    

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/loops/fit_loop.py:310: The number of training batches (9) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

[rank: 0] Metric val_loss improved by 0.004 >= min_delta = 0.0001. New best score: 0.565
[rank: 2] Metric val_loss improved by 0.004 >= min_delta = 0.0001. New best score: 0.565
[rank: 1] Metric val_loss improved by 0.004 >= min_delta = 0.0001. New best score: 0.565


Validation: |          | 0/? [00:00<?, ?it/s]

[rank: 0] Metric val_loss improved by 0.003 >= min_delta = 0.0001. New best score: 0.562
[rank: 2] Metric val_loss improved by 0.003 >= min_delta = 0.0001. New best score: 0.562
[rank: 1] Metric val_loss improved by 0.003 >= min_delta = 0.0001. New best score: 0.562


Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
single_gpu_trainer = pl.Trainer(
    devices=1,
    accelerator="auto"
)
single_gpu_trainer.test(model, datamodule=dm)